# 0. Imports

In [ ]:
import cda2
import datetime
import pyspark.sql.functions as F
import pyspark.sql.types as T
import json
import re

from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number

# 1. Connect to Spark

In [ ]:
api = cda2.Api()

Set configuration parameters to better optimize queries.

In [ ]:
config = {
    "spark.sql.adaptive.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.parallelismFirst": "false",
    "spark.sql.adaptive.coalescePartitions.minPartitionSize": "1m",
    "spark.executor.memory": "8g",
    "spark.executor.memoryOverhead": "16g",
}

Start Spark and specify number of cpus to use. 400 is quite high, but we'll be running 1 year at a time and want to have it done in just a few minutes.

In [ ]:
api.start_spark(n_executors=100, config=config)

In [ ]:
%run ./shared_variables.ipynb
%run ./S3_csv_writers.ipynb

In [ ]:
#year0 = "2025"
year1 = str(int(year0) + 1)

In [ ]:
dates = {"start_date": year0 + "-01-01", "end_date": year1 +"-01-01"}

In [ ]:
def get_alpha_portion(route_name:T.StringType) -> str:
    regex = "^([a-zA-Z]+)\d+"
    match = re.search(regex, route_name)
    if match:
        output = match.group(1)
    else:
        output = ""
    return output

get_alpha_portion_udf = F.udf(get_alpha_portion, T.StringType())

#result = get_alpha_portion("AB100")
#print(result)

In [ ]:
def get_numeric_portion(route_name:T.StringType) -> int:
    regex = "^[a-zA-Z]+(\d+)"
    match = re.search(regex, route_name)
    if match:
        output = int(match.group(1))
    else:
        output = -1
    return output

get_numeric_portion_udf = F.udf(get_numeric_portion, T.StringType())

#result = get_numeric_portion("AB100")
#print(result)

In [ ]:
def pad_seq_with_zeros(seq:T.IntegerType) -> str:
    output = str(seq).zfill(5)
    return output

pad_seq_with_zeros_udf = F.udf(pad_seq_with_zeros, T.StringType())

#result = pad_seq_with_zeros("170")
#print(result)

In [ ]:
legs_json = '{"fields":[{"metadata":{},"name":"legs","nullable":true,"type":{"containsNull":true,"elementType":{"fields":[{"metadata":{},"name":"primary_key","nullable":true,"type":"string"},{"metadata":{},"name":"path_terminator","nullable":true,"type":{"fields":[{"metadata":{},"name":"source_key","nullable":true,"type":"string"},{"metadata":{},"name":"identification","nullable":true,"type":{"fields":[{"metadata":{},"name":"name","nullable":true,"type":"string"},{"metadata":{},"name":"icao_region","nullable":true,"type":"string"},{"metadata":{},"name":"section_subsection","nullable":true,"type":"string"}],"type":"struct"}},{"metadata":{},"name":"latitude","nullable":true,"type":"double"},{"metadata":{},"name":"longitude","nullable":true,"type":"double"},{"metadata":{},"name":"elevation","nullable":true,"type":"float"},{"metadata":{},"name":"magnetic_variation","nullable":true,"type":{"fields":[{"metadata":{},"name":"published","nullable":true,"type":"float"},{"metadata":{},"name":"modeled","nullable":true,"type":"float"}],"type":"struct"}},{"metadata":{},"name":"navigation_source","nullable":true,"type":"string"}],"type":"struct"}},{"metadata":{},"name":"fix_description","nullable":true,"type":"string"},{"metadata":{},"name":"sequence_number","nullable":true,"type":"integer"},{"metadata":{},"name":"boundary_code","nullable":true,"type":"string"},{"metadata":{},"name":"level","nullable":true,"type":"string"},{"metadata":{},"name":"direction_restriction","nullable":true,"type":"string"},{"metadata":{},"name":"cruise_table_indicator","nullable":true,"type":"string"},{"metadata":{},"name":"eu_indicator","nullable":true,"type":"boolean"},{"metadata":{},"name":"recommended_navaid","nullable":true,"type":{"fields":[{"metadata":{},"name":"source_key","nullable":true,"type":"string"},{"metadata":{},"name":"identification","nullable":true,"type":{"fields":[{"metadata":{},"name":"name","nullable":true,"type":"string"},{"metadata":{},"name":"icao_region","nullable":true,"type":"string"},{"metadata":{},"name":"section_subsection","nullable":true,"type":"string"}],"type":"struct"}},{"metadata":{},"name":"latitude","nullable":true,"type":"double"},{"metadata":{},"name":"longitude","nullable":true,"type":"double"},{"metadata":{},"name":"elevation","nullable":true,"type":"float"},{"metadata":{},"name":"magnetic_variation","nullable":true,"type":{"fields":[{"metadata":{},"name":"published","nullable":true,"type":"float"},{"metadata":{},"name":"modeled","nullable":true,"type":"float"}],"type":"struct"}},{"metadata":{},"name":"navigation_source","nullable":true,"type":"string"}],"type":"struct"}},{"metadata":{},"name":"fix_guidance","nullable":true,"type":{"fields":[{"metadata":{},"name":"rnp","nullable":true,"type":"float"},{"metadata":{},"name":"theta","nullable":true,"type":"float"},{"metadata":{},"name":"rho","nullable":true,"type":"float"},{"metadata":{},"name":"outbound_magnetic_course","nullable":true,"type":"float"},{"metadata":{},"name":"inbound_magnetic_course","nullable":true,"type":"float"},{"metadata":{},"name":"computed_inbound_course","nullable":true,"type":"double"},{"metadata":{},"name":"computed_route_distance","nullable":true,"type":"double"},{"metadata":{},"name":"route_distance","nullable":true,"type":"float"},{"metadata":{},"name":"hold_time","nullable":true,"type":"integer"},{"metadata":{},"name":"overfly","nullable":true,"type":"boolean"}],"type":"struct"}},{"metadata":{},"name":"altitude_limits","nullable":true,"type":{"fields":[{"metadata":{},"name":"description","nullable":true,"type":"string"},{"metadata":{},"name":"value1","nullable":true,"type":"float"},{"metadata":{},"name":"value2","nullable":true,"type":"float"}],"type":"struct"}},{"metadata":{},"name":"maximum_altitude","nullable":true,"type":"float"},{"metadata":{},"name":"fixed_radius_turn_indicator","nullable":true,"type":"float"}],"type":"struct"},"type":"array"}}],"type":"struct"}'

In [ ]:
legs_schema = T.StructType.fromJson(json.loads(legs_json))

In [ ]:
def get_uniq_fix_names(fix_names:T.StringType, legs:legs_schema) -> str:
    uniq_fix_name_dict = {}
    uniq_fix_name_list = []
   
    for a_leg in legs:
        fix_name = a_leg.path_terminator.identification.name
        icao_code = a_leg.path_terminator.identification.icao_region
        uniq_fix_name = fix_name + "(" + icao_code + ")"
#        uniq_fix_name_dict.update({fix_name, uniq_fix_name})
        uniq_fix_name_dict[fix_name] = uniq_fix_name

    fixes = fix_names.split(':')
    for a_fix in fixes:
        uniq_fix_name_list.append(uniq_fix_name_dict[a_fix])
        
#    uniq_fix_names = F.concat_ws(":", uniq_fix_name_list)
    uniq_fix_names = ":".join(uniq_fix_name_list)
    
#    uniq_fix_names = fix_names
    
    return uniq_fix_names

get_uniq_fix_names_udf = F.udf(get_uniq_fix_names, T.StringType())

#result = pad_seq_with_zeros("170")
#print(result)

In [ ]:
def num_path_terminators(fix_names:T.StringType) -> int:
    fixes = fix_names.split(':')
    c = len(fixes)
    return c

num_path_terminators_udf = F.udf(num_path_terminators, T.IntegerType())

#result = num_path_terminators("170:150:134:999")
#print(result)

# 3. Retrieve all routes and explode the in each route fixes

In [ ]:
df_routes_raw = (
    api.dataframe("ArincAirway", **dates, metadata=True)
    .select(
        F.col("identifier").alias("route"),
        F.col("arinc_record_info.customer_area_code").alias("area_code"),
        "legs",
        F.explode("legs").alias("leg"),
        F.col("leg.path_terminator.identification.icao_region").alias("icao_region"),
        F.col("leg.path_terminator.identification.name").alias("fix_name"),
        F.col("metadata.effective_end_date").alias("end_date"),
        F.col("path_terminators").alias("fix_names"),
        "navigation_source",
    )
    .withColumn("route_uniq", F.concat("route", F.lit("("), "area_code", F.lit("/"), "icao_region", F.lit(")")))
    .withColumn("fix_name_uniq", F.concat("fix_name", F.lit("("), "icao_region", F.lit(")")))
    .withColumn("fix_names_uniq", get_uniq_fix_names_udf(F.col("fix_names"), F.col("legs")))
    .withColumn("fix_names_uniq_array", F.split(F.col("fix_names_uniq"), ":"))
    .withColumn("num_path_terminators", num_path_terminators_udf(F.col("fix_names")).cast("int"))
    .withColumn("alpha", get_alpha_portion_udf("route"))
    .withColumn("numeric", get_numeric_portion_udf("route").cast("int"))
#    .filter(F.col("route") == "Q156")
#    .filter(F.col("area_code") == "USA")
    .drop("legs", "leg")
)

#df_routes_raw.show()

In [ ]:
## get unique routes
window = Window.partitionBy("route_uniq").orderBy(col("end_date").desc())
df_unique_routes = (df_routes_raw
    .withColumn("row", row_number().over(window))
    .filter(col("row") == 1)
    .drop("row")
)
#df_unique_routes.show()

In [ ]:
df_unique_routes.select("route", "area_code", "icao_region", "fix_names_uniq", "end_date").show(truncate=False)

# 4. Save all_routes.csv

In [ ]:
df_routes_output = (
    df_unique_routes
        .select("route_uniq", "route", "alpha", "numeric", "area_code", "icao_region", "navigation_source", "fix_names", "fix_names_uniq")
        .orderBy("alpha", "numeric", "area_code", "icao_region")
)

In [ ]:
(
    df_routes_output
    .repartition(1)
    .write.option("header", True)
    .csv("CRAFT/" + year0 + "/routes/all_routes", compression="None", mode="overwrite")
)

In [ ]:
# write a Spark DataFrame to your S3 user space
# spark_write_cre(df_routes_output, path="CRAFT/" + year0 + "/routes/all_routes", writer=write_csv, compression=None, mode="overwrite")

# 5. Save all_route_fixes.csv

## first get all fixes

In [ ]:
df_fixes_raw = (
    api.dataframe("ArincFix", **dates, metadata=True)
    .select(
        F.col("identification.name").alias("fix_name"),
        F.col("arinc_record_info.customer_area_code").alias("area_code"),
        F.col("identification.icao_region").alias("icao_region"),
        "latitude",
        "longitude",
        F.col("navaid_info.dme_latitude").alias("dme_latitude"),
        F.col("navaid_info.dme_longitude").alias("dme_longitude"),
        F.col("magnetic_variation.modeled").alias("magnetic_variation"),
        F.col("metadata.effective_end_date").alias("end_date"),
    )
    .withColumn("fix_name_uniq", F.concat("fix_name", F.lit("("), "icao_region", F.lit(")")))
)
#df_fixes_raw.show()

In [ ]:
window = Window.partitionBy("fix_name_uniq").orderBy(col("end_date").desc())
df_unique_fixes = (
    df_fixes_raw
    .withColumn("row", row_number().over(window))
    .filter(col("row") == 1)
    .drop("row")
)
#df_unique_fixes.show()

## then get a list of unique fix_names from the routes

In [ ]:
df_fixes_used_in_routes = (
    df_unique_routes
    .select(
        F.explode(F.split(F.col("fix_names_uniq"), ":").alias("fix_name_uniq")).alias("fix_name_uniq"),
        "route",
        "route_uniq",
    )
    .distinct()
    .orderBy("fix_name_uniq")
)
#df_fixes_used_in_routes.show()

## now join fixes_used_in_routes with our list of fixes

In [ ]:
df_route_fixes = (
    df_unique_fixes
    .join(df_fixes_used_in_routes, on=["fix_name_uniq"], how="inner")
    .orderBy("fix_name_uniq", "route_uniq")
)
#df_route_fixes.show()

## save all_route_fixes.csv

In [ ]:
df_route_fixes_output = (
    df_route_fixes
        .groupBy("fix_name_uniq", "fix_name", "area_code", "icao_region", "latitude", "longitude", "dme_latitude", "dme_longitude", "magnetic_variation")
        .agg(F.concat_ws(":", F.collect_set("route")).alias("routes"), F.concat_ws(":", F.collect_set("route_uniq")).alias("routes_uniq_using_fix"))
        .orderBy("fix_name_uniq")
)

In [ ]:
(
    df_route_fixes_output
    .repartition(1)
    .write.option("header", True)
    .csv("CRAFT/" + year0 + "/routes/all_route_fixes", compression="None", mode="overwrite")
)

In [ ]:
# write a Spark DataFrame to your S3 user space
# spark_write_cre(df_route_fixes_output, path="CRAFT/" + year0 + "/routes/all_route_fixes", writer=write_csv, compression=None, mode="overwrite")